# Local test: filter tweets by GV dates + 6-mile radius

Local copy of `02_match_tweets_to_incidents_cluster.ipynb` for testing **before** Harvard FAS.

**Inputs**
- `data/input/gun_violence_incidents_sample_4.csv` — small incident list (date + address)
- `data/input/sample_geotweets.csv` — local tweets (tab-separated)

**Filters (same logic as production)**
1. Tweet day is incident **t**, **t−1**, or **t−2**
2. Tweet lat/lon within **4 miles** of geocoded incident

**Outputs**
- Filtered CSV: `data/output/filtered/sample_tweets_matched_to_incidents.csv`
- Map shown in notebook cell output only (not saved)


## 0. Setup


In [ ]:
from __future__ import annotations

import json
import math
import os
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_INPUT = ROOT / "data" / "input"
DATA_FILTERED = ROOT / "data" / "output" / "filtered"
DATA_LABELED = ROOT / "data" / "output" / "labeled"
DATA_CHECKPOINTS = ROOT / "data" / "output" / "checkpoints"
for _d in (DATA_INPUT, DATA_FILTERED, DATA_LABELED, DATA_CHECKPOINTS):
    _d.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

LATLONG_PATH = DATA_INPUT / "gun_violence_incidents_sample_4.csv"
TWEETS_PATH = DATA_INPUT / "sample_geotweets.csv"
GEOCODE_CACHE_PATH = DATA_INPUT / "address_to_latlon_cache.csv"
OUTPUT_DIR = DATA_FILTERED
OUTPUT_FILE = OUTPUT_DIR / "sample_tweets_matched_to_incidents.csv"

MAX_YEAR = 2023
RADIUS_MILES = 6.0
EARTH_RADIUS_MILES = 3958.7613
DAY_OFFSETS = (0, -1, -2)

print("ROOT           :", ROOT)
print("latlongtest    :", LATLONG_PATH, "exists=", LATLONG_PATH.exists())
print("tweets         :", TWEETS_PATH, "exists=", TWEETS_PATH.exists())
print("geocode cache  :", GEOCODE_CACHE_PATH, "exists=", GEOCODE_CACHE_PATH.exists())
print("output file    :", OUTPUT_FILE)
print("Radius         :", RADIUS_MILES, "miles")
print("Window         : t / t-1 / t-2")


## 1. Load `data/input/gun_violence_incidents_sample_4.csv`, geocode addresses


In [ ]:
def load_latlong_incidents(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    required = {"Incident Date", "State", "City Or County", "Address"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    df = raw.copy()
    df["Incident Date"] = pd.to_datetime(df["Incident Date"], format="mixed", errors="coerce")
    addr_raw = df["Address"]
    addr = addr_raw.astype(str).str.strip()
    missing_addr = addr_raw.isna() | (addr == "") | (addr.str.lower() == "nan")
    n_missing = int(missing_addr.sum())

    df = df.loc[~missing_addr].copy()
    df = df.dropna(subset=["Incident Date"])
    df = df[df["Incident Date"].dt.year <= MAX_YEAR].copy()
    df["Incident Date"] = df["Incident Date"].dt.normalize()
    df["Address"] = df["Address"].astype(str).str.strip()
    df["State"] = df["State"].astype(str).str.strip()
    df["City Or County"] = df["City Or County"].astype(str).str.strip()
    df["query"] = df["Address"] + ", " + df["City Or County"] + ", " + df["State"] + ", USA"
    df = df.reset_index(drop=True)

    print(f"Loaded {path.name}: kept {len(df)} / {len(raw)} (skipped missing addr={n_missing})")
    display(df)
    return df


def load_geocode_cache(cache_path: Path) -> dict:
    if not cache_path.exists():
        return {}
    cache_df = pd.read_csv(cache_path)
    out = {}
    for _, row in cache_df.iterrows():
        q = str(row["query"])
        lat, lon = row.get("lat"), row.get("lon")
        out[q] = (None, None) if pd.isna(lat) or pd.isna(lon) else (float(lat), float(lon))
    print(f"Cache loaded: {len(out)} queries")
    return out


def save_geocode_cache(cache: dict, cache_path: Path) -> None:
    rows = [{"query": q, "lat": lat, "lon": lon} for q, (lat, lon) in sorted(cache.items())]
    pd.DataFrame(rows).to_csv(cache_path, index=False)
    print(f"Cache saved: {len(rows)} → {cache_path}")


def geocode_one_arcgis(query: str, timeout: float = 15.0):
    import urllib.parse
    import urllib.request

    params = urllib.parse.urlencode(
        {"f": "json", "singleLine": query, "maxLocations": 1, "outFields": "Match_addr"}
    )
    url = (
        "https://geocode.arcgis.com/arcgis/rest/services/World/GeocodeServer/"
        f"findAddressCandidates?{params}"
    )
    req = urllib.request.Request(url, headers={"User-Agent": "gv-local-test/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    cands = payload.get("candidates") or []
    if not cands:
        return None, None
    loc = cands[0].get("location") or {}
    lat, lon = loc.get("y"), loc.get("x")
    if lat is None or lon is None:
        return None, None
    return float(lat), float(lon)


def geocode_incidents(df: pd.DataFrame, cache_path: Path) -> pd.DataFrame:
    cache = load_geocode_cache(cache_path)
    uniq = (
        df[["query", "Address", "City Or County", "State"]]
        .drop_duplicates(subset=["query"])
        .rename(columns={"Address": "street", "City Or County": "city", "State": "state"})
    )
    todo = [r for r in uniq.to_dict("records") if r["query"] not in cache or cache[r["query"]][0] is None]
    print(f"Need geocoding: {len(todo)} / {len(uniq)}")

    for i, rec in enumerate(tqdm(todo, desc="Geocoding"), start=1):
        q = rec["query"]
        try:
            cache[q] = geocode_one_arcgis(q)
        except Exception as e:
            print(f"  fail {q}: {e}")
            cache[q] = (None, None)
        time.sleep(0.15)
        if i % 20 == 0 or i == len(todo):
            save_geocode_cache(cache, cache_path)

    if not todo and cache_path.exists():
        pass
    else:
        save_geocode_cache(cache, cache_path)

    coords = [cache.get(q, (None, None)) for q in df["query"]]
    out = df.copy()
    out["incident_lat"] = [c[0] for c in coords]
    out["incident_lon"] = [c[1] for c in coords]
    n_fail = int(out["incident_lat"].isna().sum())
    out = out.dropna(subset=["incident_lat", "incident_lon"]).reset_index(drop=True)
    print(f"Geocoded incidents: {len(out)} (dropped {n_fail})")
    display(out)
    return out


INCIDENTS_DF = load_latlong_incidents(LATLONG_PATH)
INCIDENTS_DF = geocode_incidents(INCIDENTS_DF, GEOCODE_CACHE_PATH)


## 2. Build day index (t, t−1, t−2)


In [ ]:
def build_day_index(incidents: pd.DataFrame):
    allowed = set()
    day_to_incidents = {}
    for _, row in incidents.iterrows():
        t = pd.Timestamp(row["Incident Date"]).normalize()
        incident = {
            "incident_ts": t,
            "lat": float(row["incident_lat"]),
            "lon": float(row["incident_lon"]),
            "address": row["Address"],
            "city": row["City Or County"],
            "state": row["State"],
        }
        for offset in DAY_OFFSETS:
            d = (t + pd.Timedelta(days=offset)).date()
            if d.year > MAX_YEAR:
                continue
            allowed.add(d)
            day_to_incidents.setdefault(d, []).append(incident)

    years = sorted({str(d.year) for d in allowed})
    print(f"Incidents: {len(incidents)}")
    print(f"Allowed days: {len(allowed)}")
    print(f"Years: {years}")
    return allowed, day_to_incidents, years


ALLOWED_DAYS, DAY_TO_INCIDENTS, TARGET_YEARS = build_day_index(INCIDENTS_DF)


## 3. Load `data/input/sample_geotweets.csv` and apply date + 6-mile filters


In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * EARTH_RADIUS_MILES * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


def filter_tweets(tweets_path: Path) -> pd.DataFrame:
    # sample_geotweets.csv is tab-separated
    df = pd.read_csv(tweets_path, sep="\t", dtype=str, on_bad_lines="skip")
    print(f"Loaded tweets: {len(df)} from {tweets_path.name}")

    if "tweet_date" not in df.columns and "date" in df.columns:
        df["tweet_date"] = df["date"]
    elif "tweet_date" in df.columns and "date" in df.columns:
        df["tweet_date"] = df["tweet_date"].fillna(df["date"])

    if "latitude" not in df.columns or "longitude" not in df.columns:
        raise ValueError("Tweets need latitude/longitude columns")

    df["tweet_date"] = pd.to_datetime(df["tweet_date"], errors="coerce")
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["tweet_date", "latitude", "longitude"]).copy()
    df["just_date"] = df["tweet_date"].dt.date

    date_hit = df[df["just_date"].isin(ALLOWED_DAYS)].copy()
    print(f"After date filter (t/t-1/t-2): {len(date_hit)}")

    matched = []
    for just_date, group in date_hit.groupby("just_date", sort=False):
        for incident in DAY_TO_INCIDENTS.get(just_date, []):
            piece = group.copy()
            dist = haversine_miles(
                piece["latitude"].to_numpy(),
                piece["longitude"].to_numpy(),
                incident["lat"],
                incident["lon"],
            )
            keep = dist <= RADIUS_MILES
            if not np.any(keep):
                continue
            piece = piece.iloc[np.flatnonzero(keep)].copy()
            piece["distance_miles"] = dist[keep]  # useful for local test QA only
            delta = (just_date - incident["incident_ts"].date()).days
            piece["tag"] = {0: "t", -1: "t-1", -2: "t-2"}.get(delta, f"t{delta}")
            piece["matched_incident_date"] = incident["incident_ts"].strftime("%Y-%m-%d")
            piece["incident_city"] = incident["city"]
            piece["incident_lat"] = incident["lat"]
            piece["incident_lon"] = incident["lon"]
            matched.append(piece)

    if not matched:
        print("No tweets passed both filters.")
        return pd.DataFrame()

    out = pd.concat(matched, ignore_index=True)
    out["tweet_date"] = out["tweet_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
    out = out.drop(columns=["just_date"], errors="ignore")
    print(f"After 6-mile spatial filter: {len(out)}")
    if "tag" in out.columns:
        display(out["tag"].value_counts().rename("count").to_frame())
    display(out.head(10))
    return out


FILTERED_DF = filter_tweets(TWEETS_PATH)


## 4. Visualization — cell output only

Run after sections 0–3. Map appears **under this cell** (not saved anywhere).

- **Purple** = all tweets on t / t−1 / t−2  
- **Blue** = filtered tweets kept by the 6-mile ring  
- **Red** = incident + radius  


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PURPLE = "#7b2cbf"  # ALL tweets in the date window
BLUE = "#1f77b4"    # FILTERED tweets (kept: inside 6-mi ring)
RING = "#d62728"


def tweets_for_incident_window(tweets_path: Path, incident: dict) -> pd.DataFrame:
    """All tweets on t / t-1 / t-2 for one incident, with distance + filter flag."""
    df = pd.read_csv(tweets_path, sep="\t", dtype=str, on_bad_lines="skip")
    if "tweet_date" not in df.columns and "date" in df.columns:
        df["tweet_date"] = df["date"]
    elif "tweet_date" in df.columns and "date" in df.columns:
        df["tweet_date"] = df["tweet_date"].fillna(df["date"])

    df["tweet_date"] = pd.to_datetime(df["tweet_date"], errors="coerce")
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df = df.dropna(subset=["tweet_date", "latitude", "longitude"]).copy()
    df["just_date"] = df["tweet_date"].dt.date

    window_days = {
        (incident["incident_ts"] + pd.Timedelta(days=off)).date()
        for off in DAY_OFFSETS
        if (incident["incident_ts"] + pd.Timedelta(days=off)).year <= MAX_YEAR
    }
    hit = df[df["just_date"].isin(window_days)].copy()
    if hit.empty:
        return hit

    dist = haversine_miles(
        hit["latitude"].to_numpy(),
        hit["longitude"].to_numpy(),
        incident["lat"],
        incident["lon"],
    )
    hit["distance_miles"] = dist
    # Filtered = passes spatial filter (inside radius)
    hit["is_filtered"] = hit["distance_miles"] <= RADIUS_MILES
    return hit


# Map in notebook cell output ONLY — never saved to pipeline_outputs/
n = len(INCIDENTS_DF)
ncols = 2
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 5.2 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

print("Purple = ALL tweets (date window t/t-1/t-2)")
print("Blue   = FILTERED tweets (inside 6-mi ring — kept)")
print("Red    = incident + radius")
print("(Shown in cell output only — not written to disk)\n")

for ax, (_, inc) in zip(axes, INCIDENTS_DF.iterrows()):
    lat0 = float(inc["incident_lat"])
    lon0 = float(inc["incident_lon"])
    dlat = RADIUS_MILES / 69.0
    dlon = RADIUS_MILES / (69.0 * max(0.2, math.cos(math.radians(lat0))))

    incident = {
        "incident_ts": pd.Timestamp(inc["Incident Date"]).normalize(),
        "lat": lat0,
        "lon": lon0,
    }
    window_tweets = tweets_for_incident_window(TWEETS_PATH, incident)
    n_all = len(window_tweets)
    n_filt = int(window_tweets["is_filtered"].sum()) if n_all else 0
    print(
        f"{inc['City Or County']} ({pd.Timestamp(inc['Incident Date']).date()}): "
        f"all={n_all} | filtered(blue)={n_filt}"
    )

    theta = np.linspace(0, 2 * np.pi, 360)
    ax.plot(lon0 + dlon * np.cos(theta), lat0 + dlat * np.sin(theta), color=RING, lw=2.5, zorder=3)
    ax.fill(lon0 + dlon * np.cos(theta), lat0 + dlat * np.sin(theta), color=RING, alpha=0.07, zorder=1)
    ax.scatter([lon0], [lat0], c=RING, s=160, marker="*", zorder=6, edgecolors="white", linewidths=0.6)

    if not window_tweets.empty:
        # 1) ALL tweets in purple
        ax.scatter(
            window_tweets["longitude"].astype(float),
            window_tweets["latitude"].astype(float),
            c=PURPLE,
            s=28,
            alpha=0.55,
            zorder=2,
            label=f"all tweets n={n_all}",
        )
        # 2) FILTERED tweets on top in blue
        filtered = window_tweets[window_tweets["is_filtered"]]
        if not filtered.empty:
            ax.scatter(
                filtered["longitude"].astype(float),
                filtered["latitude"].astype(float),
                c=BLUE,
                s=42,
                alpha=0.95,
                zorder=5,
                edgecolors="white",
                linewidths=0.4,
                label=f"filtered n={n_filt}",
            )

    # Zoom: include all purple points if they exist, but keep ring readable
    pad = 1.35
    x0, x1 = lon0 - dlon * pad, lon0 + dlon * pad
    y0, y1 = lat0 - dlat * pad, lat0 + dlat * pad
    if not window_tweets.empty:
        # expand view if outside tweets exist, with modest padding
        lons = window_tweets["longitude"].astype(float)
        lats = window_tweets["latitude"].astype(float)
        # If points sprawl very far, still focus near ring but show a bit more context
        span_lon = max(lons.max() - lons.min(), 2 * dlon)
        span_lat = max(lats.max() - lats.min(), 2 * dlat)
        # Cap extreme zoom-out so Nashville-style outliers don't shrink the ring to a dot
        max_span_lon = 8 * dlon
        max_span_lat = 8 * dlat
        span_lon = min(span_lon, max_span_lon)
        span_lat = min(span_lat, max_span_lat)
        x0, x1 = lon0 - span_lon / 2 * 1.2, lon0 + span_lon / 2 * 1.2
        y0, y1 = lat0 - span_lat / 2 * 1.2, lat0 + span_lat / 2 * 1.2

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect(1 / max(0.2, math.cos(math.radians(lat0))))
    ax.set_title(f"{inc['City Or County']} — {pd.Timestamp(inc['Incident Date']).date()}", fontsize=12)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)

for ax in axes[n:]:
    ax.axis("off")

fig.legend(
    handles=[
        Line2D([0], [0], marker="o", color="w", markerfacecolor=PURPLE, markersize=10, label="all tweets"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=BLUE, markersize=10, label="filtered tweets"),
        Line2D([0], [0], color=RING, lw=2.5, label=f"{RADIUS_MILES}-mi ring"),
    ],
    loc="upper center",
    ncol=3,
    frameon=True,
)
plt.suptitle(
    "Visual analysis: purple=all tweets, blue=filtered (inside 6-mi ring)",
    fontsize=14,
    y=1.02,
)

display(fig)
plt.show()
plt.close(fig)


## 5. Save filtered output


In [ ]:
if FILTERED_DF is None or FILTERED_DF.empty:
    print("No tweets matched — nothing written.")
else:
    # Production-like columns: all original tweet cols + tag
    # Keep QA helpers (distance/incident_*) in this local test file for inspection
    FILTERED_DF.to_csv(OUTPUT_FILE, index=False)
    print(f"SUCCESS: {len(FILTERED_DF)} rows → {OUTPUT_FILE}")
    print("\nBy incident city:")
    display(FILTERED_DF["incident_city"].value_counts().to_frame("tweets"))
    print("\nBy tag:")
    display(FILTERED_DF["tag"].value_counts().to_frame("tweets"))
    display(FILTERED_DF.head(5))
